# Battery Cycle‑Life Analyzer — Demo

Fit degradation models, project remaining useful life, and visualise results.

### Setup for a fresh Google Colab run
If you open this notebook directly from GitHub in Colab, install bcla first:

```bash
python -m pip install --upgrade pip
python -m pip install --quiet git+https://github.com/mohammadrezwankhan/battery-cycle-life-analyzer.git
```



In [ ]:
import importlib
import subprocess
import sys

import numpy as np
import matplotlib.pyplot as plt

if 'google.colab' in sys.modules:
    try:
        import bcla  # type: ignore[import-not-found]
    except ModuleNotFoundError:
        subprocess.run(["python", "-m", "pip", "install", "--quiet", "--upgrade", "pip"], check=True)
        subprocess.run(["python", "-m", "pip", "install", "--quiet", "git+https://github.com/mohammadrezwankhan/battery-cycle-life-analyzer.git"], check=True)
        importlib.invalidate_caches()
        import bcla  # type: ignore[import-not-found]

from bcla import core, datasets, viz


## 1. Use the reproducible NMC synthetic trajectory
All synthetic data is fixed by default (seeded) for reproducible outputs.


In [ ]:
cycles, capacity = datasets.synthetic_nmc(cycles=1000, seed=7)
print(f"Cycles: {len(cycles)}, capacity range: [{capacity.min():.4f}, {capacity.max():.4f}]")

assert np.all(np.isfinite(cycles)) and np.all(np.isfinite(capacity))
assert len(cycles) == len(capacity) > 0


## 2. Fit all degradation models
This compares linear, power-law, and logarithmic models.


In [ ]:
results = core.fit_all_models(cycles, capacity)
for name, r in results.items():
    print(r.summary() + "\n")


## 3. Best model & life projection
The notebook now checks the `None` case explicitly and reports both EOL and RUL.


In [ ]:
name, best = core.best_model(results, criterion="rmse")
print(f"Best model: {name} (R² = {best.r_squared:.4f})")
eol = best.eol_cycle(eol_fraction=0.8)
current_cycle = float(cycles[-1])
if eol is None:
    print("Projected EOL (80%): outside supported projection window")
    print("Projected RUL (80%): unavailable")
else:
    rul_cycles = max(0.0, eol - current_cycle)
    print(f"Projected EOL (80%): {eol:.0f} cycles")
    print(f"Projected RUL (80%): {rul_cycles:.0f} cycles")


## 4. Visualise fitted curves


In [ ]:
fig = viz.model_comparison(results)
fig.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Temperature effect on cycle life


In [ ]:
fig2, ax = plt.subplots(figsize=(8, 4.5))
viz.eol_vs_temperature(q0=1.0, k=0.00018, temperatures=[15, 25, 35, 45, 55], ax=ax)
fig2.savefig("temperature_effect.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Compare selected chemistry models
Both are synthetic, illustrative trajectories (not validated chemistry benchmarks).


In [ ]:
x_lfp, y_lfp = datasets.synthetic_lfp(cycles=1500, seed=42)
x_nmc, y_nmc = datasets.synthetic_nmc(cycles=1000, seed=7)

chemistry_data = [("LFP", x_lfp, y_lfp), ("NMC", x_nmc, y_nmc)]

fig3, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for axis, (name, x, y) in zip((ax1, ax2), chemistry_data):
    results_cell = core.fit_all_models(x, y)
    best_name, best_result = core.best_model(results_cell, criterion="rmse")
    print(f"{name} best model: {best_name} (R²={best_result.r_squared:.4f}, RMSE={best_result.rmse:.5f})")
    viz.capacity_fade(best_result, ax=axis, title=f"{name}: best = {best_name}")

    eol = best_result.eol_cycle(eol_fraction=0.8)
    if eol is not None:
        print(f"{name} projected 80% EOL (selected model): {eol:.0f} cycles")
    else:
        print(f"{name} 80% EOL (selected model): outside projection window")

fig3.suptitle("Chemistry Comparison", fontsize=14, y=1.03)
fig3.tight_layout()
fig3.savefig("chemistry_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
